## Inference patterns reference

### Purpose

This notebook serves as a reference guide for supported inference patterns using models registered through the BYOM workflow.

It provides implementation examples for:

- Batch inference (multiple execution strategies)
- Real-time serving
- SQL-based scoring

Use this notebook as a technical reference when adapting inference logic for customer-specific architectures.

This notebook is not part of the core governance pipeline.

### Preconditions

Before using any pattern in this notebook:

- The model must be registered in Unity Catalog
- The `Champion` alias must be assigned (unless explicitly testing another version)
- Input data must match the model signature

In [ ]:
%pip install -q threadpoolctl>=3.5.0 xgboost
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

### Setup: model and tables

Run this once. All batch options below assume these variables exist.


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

catalog = "pcl"
schema = "byo_model"
model_name = "pyfunc_xgb"
input_table = f"{catalog}.{schema}.iris_features"
output_table = f"{catalog}.{schema}.iris_predictions"
registered_model_name = f"{catalog}.{schema}.{model_name}"

champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
champion_version = int(champion_mv.version)
assert champion_mv.tags.get("approval") == "approved", f"{registered_model_name}@Champion not approved"

model_uri = f"models:/{registered_model_name}@Champion"
model_info = mlflow.models.get_model_info(model_uri)
feature_cols = [inp.name for inp in model_info.signature.inputs]
id_cols = ["id"]

print(f"Model: {model_uri} (v{champion_version})")


### Run batch inference — alternative implementations

Pick **one** of the options below. Each is a **full replacement** for the "Run batch inference" code cell in **4b_batch_inference.ipynb**: same inputs (widgets + Champion resolution) and same outputs (write to `output_table`, set task values, print). Copy the chosen option’s code cell into 4b to switch inference style.


#### Option 1 — Pandas (Single-Node)

- Suitable for small datasets
- Local execution
- Simple and direct

Not recommended for production-scale workloads.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
champion_version = int(champion_mv.version)
if champion_mv.tags.get("approval") != "approved":
    raise ValueError(f"Champion model version {champion_version} is not approved")
model_uri = f"models:/{registered_model_name}@Champion"
model_info = mlflow.models.get_model_info(model_uri)
feature_cols = [inp.name for inp in model_info.signature.inputs]
id_cols = ["id"]

input_df = spark.table(input_table)
model_udf = mlflow.pyfunc.spark_udf(spark=spark, model_uri=model_uri, env_manager="virtualenv")
input_df = input_df.repartition(spark.sparkContext.defaultParallelism * 2)

scored_df = input_df.withColumn("prediction", model_udf(*[F.col(c) for c in feature_cols])).select(*id_cols, "prediction")
scored_df = scored_df.withColumn("model_version", F.lit(champion_version)).withColumn("prediction_timestamp", F.current_timestamp().cast("string"))
scored_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)
num_predictions = scored_df.count()
dbutils.jobs.taskValues.set(key="output_table", value=output_table)
dbutils.jobs.taskValues.set(key="num_predictions", value=str(num_predictions))
dbutils.jobs.taskValues.set(key="model_version", value=str(champion_version))
print(f"Batch inference completed: {registered_model_name} v{champion_version} -> {output_table} ({num_predictions} rows)")

#### Option 2 — Spark UDF (Distributed) (default in `4b_batch_inference.ipynb`)

- Scales across Spark cluster
- Recommended for large datasets
- Suitable for scheduled batch pipelines

Best general-purpose production pattern.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
champion_version = int(champion_mv.version)
if champion_mv.tags.get("approval") != "approved":
    raise ValueError(f"Champion model version {champion_version} is not approved")

model_uri = f"models:/{registered_model_name}@Champion"
model_info = mlflow.models.get_model_info(model_uri)
feature_cols = [inp.name for inp in model_info.signature.inputs]
id_cols = ["id"]

input_df = spark.table(input_table)
model_udf = mlflow.pyfunc.spark_udf(spark=spark, model_uri=model_uri, result_type="long", env_manager="virtualenv")
input_df = input_df.repartition(spark.sparkContext.defaultParallelism * 2)

scored_df = input_df.withColumn("prediction", model_udf(*[F.col(c) for c in feature_cols])).select(*id_cols, "prediction")
scored_df = scored_df.withColumn("model_version", F.lit(champion_version)).withColumn("prediction_timestamp", F.current_timestamp().cast("string"))
scored_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)
num_predictions = scored_df.count()

dbutils.jobs.taskValues.set(key="output_table", value=output_table)
dbutils.jobs.taskValues.set(key="num_predictions", value=str(num_predictions))
dbutils.jobs.taskValues.set(key="model_version", value=str(champion_version))

print(f"Batch inference completed: {registered_model_name} v{champion_version} -> {output_table} ({num_predictions} rows)")

#### Option 3 — Feature Store API

- Performs feature lookups and scoring in a single flow
- Useful when model depends on managed feature tables
- Preserves feature lineage

Use when customer architecture includes Feature Store.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
champion_version = int(champion_mv.version)
if champion_mv.tags.get("approval") != "approved":
    raise ValueError(f"Champion model version {champion_version} is not approved")
model_uri = f"models:/{registered_model_name}@Champion"
model_info = mlflow.models.get_model_info(model_uri)
feature_cols = [inp.name for inp in model_info.signature.inputs]
id_cols = ["id"]

fe = FeatureEngineeringClient()
input_df = spark.table(input_table)
scored_df = fe.score_batch(model_uri=model_uri, df=input_df).select(*id_cols, "prediction")
scored_df = scored_df.withColumn("model_version", F.lit(champion_version)).withColumn("prediction_timestamp", F.current_timestamp().cast("string"))
scored_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)
num_predictions = scored_df.count()
dbutils.jobs.taskValues.set(key="output_table", value=output_table)
dbutils.jobs.taskValues.set(key="num_predictions", value=str(num_predictions))
dbutils.jobs.taskValues.set(key="model_version", value=str(champion_version))
print(f"Batch inference completed: {registered_model_name} v{champion_version} -> {output_table} ({num_predictions} rows)")


#### Option 4 — `ai_query()` (SQL-Based Scoring)

- Requires serving endpoint (per `4a_model_serving.ipynb`)
- Enables scoring directly in SQL
- Useful for BI-driven or SQL-first workflows

Appropriate when teams operate primarily in SQL environments.


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
champion_version = int(champion_mv.version)
if champion_mv.tags.get("approval") != "approved":
    raise ValueError(f"Champion model version {champion_version} is not approved")
model_info = mlflow.models.get_model_info(f"models:/{registered_model_name}@Champion")
feature_cols = [inp.name for inp in model_info.signature.inputs]
id_cols = ["id"]
endpoint_name = f"{model_name}-endpoint"
request_cols = ", ".join(feature_cols)
sql = f"""
CREATE OR REPLACE TABLE {output_table} AS
SELECT
  id,
  ai_query(endpoint => '{endpoint_name}', request => struct({request_cols}), returnType => 'DOUBLE') AS prediction,
  {champion_version} AS model_version,
  cast(current_timestamp() AS string) AS prediction_timestamp
FROM {input_table}
"""
spark.sql(sql)
num_predictions = spark.table(output_table).count()
dbutils.jobs.taskValues.set(key="output_table", value=output_table)
dbutils.jobs.taskValues.set(key="num_predictions", value=str(num_predictions))
dbutils.jobs.taskValues.set(key="model_version", value=str(champion_version))
print(f"Batch inference completed: {registered_model_name} v{champion_version} -> {output_table} ({num_predictions} rows)")


### Model serving (real-time endpoint)

To create or update the Databricks Model Serving endpoint, use the production artifact **4a_model_serving.ipynb**. That notebook resolves Champion, verifies approval, creates/updates the endpoint, waits for READY, and sets deployment task values.


### Real-time inference (call endpoint)

Example: call the endpoint once the endpoint exists (after running 4a). Requires secret `serving_demo.pat_token` or use your token.


In [ ]:
from databricks.sdk import WorkspaceClient
import requests, json

w = WorkspaceClient()
host = w.config.host.rstrip("/")
token = dbutils.secrets.get("serving_demo", "pat_token")  # or use your PAT
endpoint_name = f"{model_name}-endpoint"
cols = feature_cols
sample = dict(zip(cols, [5.1, 3.5, 1.4, 0.2, 1.4571, 7.0, 17.85, 0.28, 6.5, 3.7, 3.7, 3.3, 7.14, 0.70, 1]))
resp = requests.post(
    f"{host}/serving-endpoints/{endpoint_name}/invocations",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"dataframe_records": [sample]},
    timeout=30
)
print(resp.status_code, resp.text)
